# Benchmark: TabPFN vs. State-of-the-Art Algorithms

Comparison of **TabPFN** against standard ML baselines on the heart failure prediction task (3-class: early / late / healthy) using identical data pipeline (`load_final_data` → `preprocess_data` → `balance_data`).

### Algorithms
| Model | Type | Why |
|-------|------|-----|
| DummyClassifier | Baseline | Lower bound — shows what random guessing achieves |
| LogisticRegression | Linear | Classic baseline, interpretable, fast |
| RandomForest | Ensemble (Bagging) | Strong default, handles categoricals via encoding |
| XGBoost | Ensemble (Boosting) | State-of-the-art for tabular data |
| TabPFN | Foundation Model | Our main model — zero-shot Bayesian inference |

### Evaluation
- **Metrics**: Accuracy, F1 Macro, ROC-AUC (OvR), per-class F1
- **Robustness**: Each model trained on 20 balanced subsets (same seeds as TabPFN_v4)
- **Fair comparison**: Same train/val/test split, same features, same balancing

In [1]:
# Imports
import sys
sys.path.insert(0, '..')

import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, recall_score, precision_score,
    classification_report
)

from fs_thesis.data_loader import load_final_data
from fs_thesis.preprocessing import preprocess_data, balance_data, get_X_y

warnings.filterwarnings('ignore')

In [2]:
# ── Run-Ordner ──
_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_{_run_timestamp}")
PLOTS_DIR = RUN_DIR / "plots"
RESULTS_DIR = RUN_DIR / "results"
for d in [PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_plot_counter = 0

def show_and_save(fig, name: str = None, width=800, height=500):
    """fig.show() + PNG speichern."""
    global _plot_counter
    _plot_counter += 1
    filename = name or f"plot_{_plot_counter:02d}"
    path = PLOTS_DIR / f"{filename}.png"
    fig.write_image(str(path), scale=3, width=width, height=height)
    print(f"💾 {path}")
    fig.show()

print(f"📁 Run-Ordner: {RUN_DIR}")

📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711


In [3]:
import logging

log_file = RUN_DIR / "run.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
log = logging.getLogger()
log.info(f"📁 Run-Ordner: {RUN_DIR}")


21:27:11 | 📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711


# 1. Data Pipeline (identical to TabPFN_v4)

In [4]:
from sklearn.model_selection import train_test_split

df = load_final_data()
df_train, df_val, df_test = preprocess_data(df)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

_, X_val_small, _, y_val_small = train_test_split(
    X_val, y_val, test_size=3000, stratify=y_val, random_state=42
)
X_val_small = X_val_small.reset_index(drop=True)

log.info(f"Val: {len(y_val)} | Val (TabPFN subsample): {len(y_val_small)} | Test: {len(y_test)}")
log.info(f"Class distribution (val):       {np.bincount(y_val)}")
log.info(f"Class distribution (val_small): {np.bincount(y_val_small)}")

21:27:12 | Val: 35753 | Val (TabPFN subsample): 3000 | Test: 44691
21:27:12 | Class distribution (val):       [ 2410  1485 31858]
21:27:12 | Class distribution (val_small): [ 202  125 2673]


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)


# 2. Define Models

All models use the **same interface**: `fit(X_train, y_train)` → `predict(X_val)` / `predict_proba(X_val)`.

For tree-based and linear models, categorical features are **one-hot encoded** (TabPFN handles them natively). The encoding is applied inside the benchmark loop.

In [5]:
# Model Definitions
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# Feature Types
FEATURE_COLS = [
    "gender", "anchor_age", "insurance", "language", "marital_status", "race", "admission_type", "bmi"
]
CAT_COLS = [
    "gender", "insurance", "language", "marital_status", "race", "admission_type"
]
NUM_COLS = ["anchor_age", "bmi"]

# Preprocessing Pipeline for sklearn Models
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), NUM_COLS),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CAT_COLS),
    ],
    remainder='drop'
)

# Model Dictionary
MODELS = {
    "DummyClassifier": Pipeline([
        ('prep', preprocessor),
        ('clf', DummyClassifier(strategy='stratified', random_state=42))
    ]),
    "LogisticRegression": Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42))
    ]),
    "RandomForest": Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            objective='multi:softprob', num_class=3,
            random_state=42, n_jobs=-1, verbosity=0,
            eval_metric='mlogloss'
        ))
    ]),
    #"TabPFN": None (extra in cell 10)
}

log.info(f"Models: {list(MODELS.keys())}")
log.info(f"Features: {len(FEATURE_COLS)} ({len(NUM_COLS)} numerical, {len(CAT_COLS)} categorical)")

21:27:12 | Models: ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost']
21:27:12 | Features: 8 (2 numerical, 6 categorical)


In [6]:
log.info(len(y_val))

21:27:12 | 35753


# 3. Robustness Benchmark Loop

Each model is trained **20 times** with different balanced training subsets (seeds 42–61), identical to the TabPFN_v4 robustness loop. This measures performance **and** stability.

In [7]:
from sklearn.base import clone

N_LOOPS = 20
N_SAMPLES = 300
log.info(f"RUN gestartet | N_LOOPS={N_LOOPS} | N_SAMPLES={N_SAMPLES}")

config = {"n_loops": N_LOOPS, "n_samples": N_SAMPLES,
          "models": list(MODELS.keys()) + ["TabPFN", "TabICL", "RF_tuned", "XGB_tuned"],
          "run_dir": str(RUN_DIR)}
json.dump(config, open(RUN_DIR / "config.json", "w"), indent=2)



all_results = []
start_total = time.time()

21:27:12 | RUN gestartet | N_LOOPS=20 | N_SAMPLES=300


In [8]:
import subprocess, json
tabicl_result_path = str(RESULTS_DIR / "tabicl_result.json")

log.info("Starte TabICL subprocess...")
proc_icl = subprocess.run(
    [sys.executable, "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabicl.py", tabicl_result_path],
    capture_output=False
)

if proc_icl.returncode == 0:
    tabicl_result = json.load(open(tabicl_result_path))
    all_results.append(tabicl_result)
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
    log.info(f"✅ TabICL: F1={tabicl_result['f1_macro']:.4f}")
else:
    log.error("❌ TabICL subprocess fehlgeschlagen")

21:27:12 | Starte TabICL subprocess...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting...
proba...
✅ F1=0.3874 | AUC=0.7997 | 10.2s


21:27:25 | ✅ TabICL: F1=0.3874


In [9]:


tabpfn_result_path = str(RESULTS_DIR / "tabpfn_result.json")

log.info("Starte TabPFN subprocess (isoliert von MPS)...")
proc = subprocess.run(
    [sys.executable, "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabpfn.py", tabpfn_result_path],
    capture_output=False
)

if proc.returncode == 0:
    tabpfn_result = json.load(open(tabpfn_result_path))
    all_results.append(tabpfn_result)
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
    log.info(f"✅ TabPFN geladen: F1={tabpfn_result['f1_macro']:.4f}")
else:
    log.error("❌ TabPFN subprocess fehlgeschlagen")

21:27:25 | Starte TabPFN subprocess (isoliert von MPS)...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting...
proba...
✅ F1=0.3766 | AUC=0.7880 | 21.9s


21:27:50 | ✅ TabPFN geladen: F1=0.3766


In [10]:
from sklearn.base import clone

for model_name, pipeline in MODELS.items():
    log.info(f"\n{'='*60}")
    log.info(f"  {model_name}")
    log.info(f"{'='*60}")
    t0 = time.time()
    model_results = []

    for i in tqdm(range(N_LOOPS), desc=model_name):
        try:
            seed = 42 + i
            df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=seed)
            X_tr, y_tr = get_X_y(df_bal)
            clf = clone(pipeline)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_val)
            y_proba = clf.predict_proba(X_val)
            f1_pc = f1_score(y_val, y_pred, average=None)
            result = {
                'model': model_name, 'run_id': i, 'seed': seed,
                'accuracy': accuracy_score(y_val, y_pred),
                'f1_macro': f1_score(y_val, y_pred, average='macro'),
                'roc_auc_macro': roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro'),
                'recall_macro': recall_score(y_val, y_pred, average='macro'),
                'precision_macro': precision_score(y_val, y_pred, average='macro'),
                'f1_class_0_early': f1_pc[0],
                'f1_class_1_late': f1_pc[1],
                'f1_class_2_healthy': f1_pc[2],
                'time_sec': time.time() - t0
            }
            all_results.append(result)
            model_results.append(result)
        except Exception as e:
            log.error(f"  ⚠️ ERROR Run {i}: {e}")

    elapsed = time.time() - t0
    if model_results:
        df_m = pd.DataFrame(model_results)
        log.info(f"  ✅ F1={df_m['f1_macro'].mean():.4f}±{df_m['f1_macro'].std():.4f} | AUC={df_m['roc_auc_macro'].mean():.4f} | {elapsed:.1f}s")
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)

log.info(f"\n{'='*60}")
log.info(f"  DONE | {len(all_results)} total runs")
log.info(f"{'='*60}")

21:27:50 | 
21:27:50 |   DummyClassifier
21:27:50 | ============================================================


DummyClassifier:   0%|          | 0/20 [00:00<?, ?it/s]

21:27:53 |   ✅ F1=0.2233±0.0000 | AUC=0.5004 | 2.1s
21:27:53 | 
21:27:53 |   LogisticRegression
21:27:53 | ============================================================


LogisticRegression:   0%|          | 0/20 [00:00<?, ?it/s]

21:27:55 |   ✅ F1=0.3779±0.0049 | AUC=0.7592 | 2.3s
21:27:55 | 
21:27:55 |   RandomForest
21:27:55 | ============================================================


RandomForest:   0%|          | 0/20 [00:00<?, ?it/s]

21:28:01 |   ✅ F1=0.3907±0.0079 | AUC=0.7882 | 6.5s
21:28:01 | 
21:28:01 |   XGBoost
21:28:01 | ============================================================


XGBoost:   0%|          | 0/20 [00:00<?, ?it/s]

21:28:14 |   ✅ F1=0.3830±0.0095 | AUC=0.7695 | 13.0s
21:28:14 | 
21:28:14 |   DONE | 82 total runs
21:28:14 | ============================================================


# 3b. Tuned Baselines: RF & XGBoost (Full Training + HPO)

Extends Section 3 with two upper-bound baselines.

**Why**: The default RF/XGBoost above use 300 balanced samples — fair for comparison  
with TabPFN, but not the full potential of these models.

**How**:
- HPO via `RandomizedSearchCV` on a stratified 20k subsample (fast)
- Final model refit on full training set (143k rows)
- Single evaluation on val set → appended to `all_results` for comparison

**Time budget**: ~30 min total (15 min per model)

In [11]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint
from sklearn.base import clone
 
HPO_SUBSAMPLE_SIZE = 20_000
HPO_N_ITER         = 30
HPO_CV             = 3
 
X_train_full, y_train_full = get_X_y(df_train)
 
_, X_hpo, _, y_hpo = train_test_split(
    X_train_full, y_train_full,
    test_size=HPO_SUBSAMPLE_SIZE,
    stratify=y_train_full,
    random_state=42,
)
log.info(f"HPO subsample: {X_hpo.shape[0]} rows | Full train: {X_train_full.shape[0]} rows")
 
 
def run_tuned_model(
    name: str,
    base_pipeline: Pipeline,
    param_dist: dict,
    sample_weight_train=None,
    sample_weight_hpo=None,
    eval_data=None,            # ← NEW: tuple (X_eval, y_eval) for loss curve
) -> tuple[dict, RandomizedSearchCV, Pipeline, dict]:  # ← CHANGED: 4-tuple
    """HPO on subsample → refit on full train → single val evaluation."""
    log.info(f"\n{'='*60}\n  {name}\n{'='*60}")
    t0 = time.time()
 
    # 1. Hyperparameter search on subsample
    search = RandomizedSearchCV(
        clone(base_pipeline),
        param_distributions=param_dist,
        n_iter=HPO_N_ITER,
        cv=HPO_CV,
        scoring='f1_macro',
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_hpo, y_hpo, clf__sample_weight=sample_weight_hpo)
    log.info(f"  HPO done | CV F1: {search.best_score_:.4f} | params: {search.best_params_}")
 
    # 2. Refit with best params on full training data
    final_model = clone(base_pipeline).set_params(**search.best_params_)
    final_model.fit(X_train_full, y_train_full, clf__sample_weight=sample_weight_train)
 
    # 2b. ← NEW: refit XGB clf with eval_set to capture training history
    # The preprocessor is already fitted from step 2, so transform() works directly.
    evals_result = {}
    if eval_data is not None:
        X_tr_proc  = final_model[:-1].transform(X_train_full)
        X_val_proc = final_model[:-1].transform(eval_data[0])
        final_model['clf'].fit(
            X_tr_proc, y_train_full,
            sample_weight=sample_weight_train,
            eval_set=[
                (X_tr_proc,  y_train_full),  # index 0 → train
                (X_val_proc, eval_data[1]),   # index 1 → val
            ],
            verbose=False,
        )
        evals_result = final_model['clf'].evals_result()
        log.info(f"  Training history captured: {len(evals_result['validation_0']['mlogloss'])} rounds")
 
    # 3. Evaluate on val set
    y_pred  = final_model.predict(X_val)
    y_proba = final_model.predict_proba(X_val)
    f1_pc   = f1_score(y_val, y_pred, average=None)
    elapsed = time.time() - t0
 
    result = {
        'model':              name,
        'run_id':             0,
        'seed':               42,
        'accuracy':           accuracy_score(y_val, y_pred),
        'f1_macro':           f1_score(y_val, y_pred, average='macro'),
        'roc_auc_macro':      roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro'),
        'recall_macro':       recall_score(y_val, y_pred, average='macro'),
        'precision_macro':    precision_score(y_val, y_pred, average='macro'),
        'f1_class_0_early':   f1_pc[0],
        'f1_class_1_late':    f1_pc[1],
        'f1_class_2_healthy': f1_pc[2],
        'time_sec':           elapsed,
    }
    log.info(f"  ✅ F1={result['f1_macro']:.4f} | AUC={result['roc_auc_macro']:.4f} | {elapsed:.0f}s")
    return result, search, final_model, evals_result  # ← CHANGED: 4-tuple
 
 
# ── Random Forest ─────────────────────────────────────────────────────────────
RF_PARAM_DIST = {
    'clf__n_estimators':     [100, 200, 300, 500],
    'clf__max_depth':        [5, 10, 15, 20, None],
    'clf__min_samples_leaf': randint(1, 10),
    'clf__max_features':     ['sqrt', 'log2'],
    'clf__class_weight':     ['balanced', 'balanced_subsample'],
}
 
rf_tuned_result, rf_search, rf_final, _ = run_tuned_model(  # ← CHANGED: unpack 4
    name          = "RF_tuned",
    base_pipeline = Pipeline([
        ('prep', clone(preprocessor)),
        ('clf',  RandomForestClassifier(random_state=42, n_jobs=-1)),
    ]),
    param_dist    = RF_PARAM_DIST,
    # no eval_data — RF has no iterative training history
)
all_results.append(rf_tuned_result)
pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
 
 
# ── XGBoost ───────────────────────────────────────────────────────────────────
XGB_PARAM_DIST = {
    'clf__n_estimators':     [200, 300, 500],
    'clf__max_depth':        [4, 6, 8],
    'clf__learning_rate':    [0.01, 0.05, 0.1],
    'clf__subsample':        [0.7, 0.8, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 1.0],
    'clf__min_child_weight': [1, 3, 5],
}
 
xgb_tuned_result, xgb_search, xgb_final, xgb_evals = run_tuned_model(  # ← CHANGED: unpack 4
    name          = "XGB_tuned",
    base_pipeline = Pipeline([
        ('prep', clone(preprocessor)),
        ('clf',  XGBClassifier(
            objective='multi:softprob', num_class=3,
            random_state=42, n_jobs=-1, verbosity=0,
            eval_metric='mlogloss',
        )),
    ]),
    param_dist          = XGB_PARAM_DIST,
    sample_weight_train = compute_sample_weight('balanced', y_train_full),
    sample_weight_hpo   = compute_sample_weight('balanced', y_hpo),
    eval_data           = (X_val, y_val),  # ← NEW: capture training history
)
all_results.append(xgb_tuned_result)
pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
 
# Persist models to disk — avoids full retraining on kernel restart
import joblib
joblib.dump(rf_final,  RESULTS_DIR / "rf_final.pkl")
joblib.dump(xgb_final, RESULTS_DIR / "xgb_final.pkl")
log.info("✅ Models saved to disk.")
 

21:28:14 | HPO subsample: 20000 rows | Full train: 143008 rows
21:28:14 | 
  RF_tuned


Fitting 3 folds for each of 30 candidates, totalling 90 fits


21:28:43 |   HPO done | CV F1: 0.4538 | params: {'clf__class_weight': 'balanced', 'clf__max_depth': 15, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__n_estimators': 500}
21:28:53 |   ✅ F1=0.4267 | AUC=0.8074 | 39s
21:28:53 | 
  XGB_tuned


Fitting 3 folds for each of 30 candidates, totalling 90 fits


21:29:25 |   HPO done | CV F1: 0.4527 | params: {'clf__subsample': 0.8, 'clf__n_estimators': 300, 'clf__min_child_weight': 1, 'clf__max_depth': 8, 'clf__learning_rate': 0.05, 'clf__colsample_bytree': 0.8}
21:29:37 |   Training history captured: 300 rounds
21:29:37 |   ✅ F1=0.4154 | AUC=0.8144 | 44s
21:29:37 | ✅ Models saved to disk.


# 3c. Plot tuned Training

In [12]:
# ── HPO Bar Charts ─────────────────────────────────────────────────────────────
def plot_hpo_results(search: RandomizedSearchCV, name: str, final_val_f1: float) -> go.Figure:
    """
    Bar chart of all 30 HPO candidate scores sorted by rank.
 
    How to read this:
    - X-axis: candidate rank (0 = best params found, 29 = worst)
    - Y-axis: mean CV F1 Macro on 20k subsample (3-fold)
    - Error bars: std across the 3 folds — measures CV stability
    - Green line: actual val F1 after refit on full 143k rows
    - Y-axis is zoomed to the actual score range (not starting at 0)
 
    What to look for:
    - Clear drop from rank 0 → rest means HPO found a genuinely better region
    - Tight error bars mean the CV is stable and trustworthy
    - Gap between green line and best bar = effect of more training data
    """
    df_cv = pd.DataFrame(search.cv_results_).sort_values('mean_test_score', ascending=False)
    df_cv['candidate'] = range(len(df_cv))
 
    y_min = df_cv['mean_test_score'].min() - df_cv['std_test_score'].max() - 0.005
    y_max = df_cv['mean_test_score'].max() + df_cv['std_test_score'].max() + 0.02
 
    best_params_str = " | ".join(
        f"{k.replace('clf__', '')}={v}" for k, v in search.best_params_.items()
    )
 
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=df_cv['candidate'],
        y=df_cv['mean_test_score'],
        error_y=dict(type='data', array=df_cv['std_test_score'].values, visible=True),
        marker_color=['#e74c3c' if i == 0 else '#3498db' for i in range(len(df_cv))],
        name='CV F1 Macro (3-fold, 20k rows)',
    ))
    fig.add_hline(
        y=final_val_f1,
        line_dash='dash', line_color='#2ecc71', line_width=2,
        annotation_text=f'Final val F1 after refit on 143k: {final_val_f1:.3f}',
        annotation_position='top left',
    )
    fig.update_layout(
        title=f'{name} — HPO Search Results (30 candidates, 3-fold CV)<br>'
              f'<sup>Best params: {best_params_str}</sup>',
        xaxis_title='Candidate Rank (0 = best)',
        yaxis_title='CV F1 Macro',
        yaxis=dict(range=[y_min, y_max]),
        template='plotly_white',
        height=450,
        showlegend=False,
    )
    return fig
 
 
fig_rf_hpo  = plot_hpo_results(rf_search,  "RF_tuned",  rf_tuned_result['f1_macro'])
fig_xgb_hpo = plot_hpo_results(xgb_search, "XGB_tuned", xgb_tuned_result['f1_macro'])
 
show_and_save(fig_rf_hpo,  "hpo_rf_search_results",  height=450)
show_and_save(fig_xgb_hpo, "hpo_xgb_search_results", height=450)
 
 
# ── XGBoost Training History (mlogloss per Boosting Round) ────────────────────
def plot_xgb_loss_curve(evals_result: dict, best_params: dict) -> go.Figure:
    """
    mlogloss per boosting round on train and val set.
    Standard diagnostic for XGBoost — shows whether the model converges
    and whether early stopping would have helped.
    """
    train_loss = evals_result['validation_0']['mlogloss']
    val_loss   = evals_result['validation_1']['mlogloss']
    rounds     = list(range(len(train_loss)))
    best_round = int(np.argmin(val_loss))
    n_estimators = best_params.get('clf__n_estimators', len(rounds))
 
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rounds, y=train_loss, mode='lines',
        name='Train mlogloss',
        line=dict(color='#e67e22', width=2),
    ))
    fig.add_trace(go.Scatter(
        x=rounds, y=val_loss, mode='lines',
        name='Val mlogloss',
        line=dict(color='#c0392b', width=2),
    ))
    fig.add_vline(
        x=best_round,
        line_dash='dash', line_color='#2ecc71', line_width=1.5,
        annotation_text=f'Best val round: {best_round}',
        annotation_position='top right',
    )
    fig.update_layout(
        title=f'XGB_tuned — Training History<br>'
              f'<sup>mlogloss per boosting round | n_estimators={n_estimators} | '
              f'lr={best_params.get("clf__learning_rate")} | depth={best_params.get("clf__max_depth")}</sup>',
        xaxis_title='Boosting Round',
        yaxis_title='mlogloss (lower = better)',
        template='plotly_white',
        height=420,
        legend=dict(x=0.75, y=0.95),
    )
    return fig
 
 
fig_xgb_loss = plot_xgb_loss_curve(xgb_evals, xgb_search.best_params_)
show_and_save(fig_xgb_loss, "xgb_loss_curve", height=420)
 
log.info("✅ Training diagnostics done.")
 

21:29:38 | Chromium init'ed with kwargs {}
21:29:38 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:29:38 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_3p_8tzg.
21:29:38 | Opening browser.
21:29:38 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpe4xdysb4.
21:29:38 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpe4xdysb4
21:29:38 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_3p_8tzg/index.html
21:29:38 | Waiting on all navigates
21:29:38 | All navigates done, putting them all in queue.
21:29:39 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_3p_8tzg/index.html
21:29:39 | Waiting on all navigates
21:29:40 | All navigates done, putting them all in queue.
21:29:40 | Tab ready: 9CD69676CBDA6E85CBE4C5589D80FB75
21:29:40 | Getting tab from queue (has 1)
21:29:40 | Got 9CD6
21:29:40 | Processing RF_tune

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/hpo_rf_search_results.png


21:29:40 | Chromium init'ed with kwargs {}
21:29:40 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:29:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpd8ciero0.
21:29:40 | Opening browser.
21:29:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp44jd8htg.
21:29:40 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp44jd8htg
21:29:40 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpd8ciero0/index.html
21:29:40 | Waiting on all navigates
21:29:40 | All navigates done, putting them all in queue.
21:29:42 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpd8ciero0/index.html
21:29:42 | Waiting on all navigates
21:29:42 | All navigates done, putting them all in queue.
21:29:42 | Tab ready: 9A0663FF002D996E44D06AB9A0AA58A7
21:29:42 | Getting tab from queue (has 1)
21:29:42 | Got 9A06
21:29:42 | Processing XGB_tun

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/hpo_xgb_search_results.png


21:29:42 | TemporaryDirectory.cleanup() worked.
21:29:42 | shutil.rmtree worked.
21:29:42 | TemporaryDirectory.cleanup() worked.
21:29:42 | shutil.rmtree worked.
21:29:42 | Chromium init'ed with kwargs {}
21:29:42 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:29:42 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpo5c4g5bo.
21:29:42 | Opening browser.
21:29:42 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_nsh_g7q.
21:29:42 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_nsh_g7q
21:29:43 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpo5c4g5bo/index.html
21:29:43 | Waiting on all navigates
21:29:43 | All navigates done, putting them all in queue.
21:29:44 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpo5c4g5bo/index.html
21:29:44 | Waiting on all navigates
21:29:45 | All navigates done, putting the

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/xgb_loss_curve.png


21:29:45 | ✅ Training diagnostics done.


# 4. Results & Comparison

In [13]:
# ── Summary Table ──
df_results = pd.DataFrame(all_results)
df_results.to_csv(RESULTS_DIR / "benchmark_all_runs.csv", index=False)

df_summary = df_results.groupby('model').agg(
    f1_mean=('f1_macro', 'mean'), f1_std=('f1_macro', 'std'),
    auc_mean=('roc_auc_macro', 'mean'), auc_std=('roc_auc_macro', 'std'),
    acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
    recall_mean=('recall_macro', 'mean'),
    precision_mean=('precision_macro', 'mean'),
    f1_early_mean=('f1_class_0_early', 'mean'), f1_early_std=('f1_class_0_early', 'std'),
    f1_late_mean=('f1_class_1_late', 'mean'), f1_late_std=('f1_class_1_late', 'std'),
    f1_healthy_mean=('f1_class_2_healthy', 'mean'), f1_healthy_std=('f1_class_2_healthy', 'std'),
    n_runs=('run_id', 'count'),
).reset_index().sort_values('f1_mean', ascending=False)

df_summary.to_csv(RESULTS_DIR / "benchmark_summary.csv", index=False)

# Schöne Darstellung
log.info("\n📊 Benchmark Summary (sorted by F1 Macro):\n")
display_cols = ['model', 'f1_mean', 'f1_std', 'auc_mean', 'auc_std', 'acc_mean', 'recall_mean', 'precision_mean']
log.info(df_summary[display_cols].to_string(index=False, float_format='{:.4f}'.format))

21:29:45 | 
📊 Benchmark Summary (sorted by F1 Macro):

21:29:45 |              model  f1_mean  f1_std  auc_mean  auc_std  acc_mean  recall_mean  precision_mean
          RF_tuned   0.4267     NaN    0.8074      NaN    0.6411       0.6169          0.4290
         XGB_tuned   0.4154     NaN    0.8144      NaN    0.6072       0.6340          0.4291
      RandomForest   0.3907  0.0079    0.7882   0.0044    0.5739       0.6008          0.4170
            TabICL   0.3874     NaN    0.7997      NaN    0.5507       0.6204          0.4226
           XGBoost   0.3830  0.0095    0.7695   0.0063    0.5712       0.5800          0.4103
LogisticRegression   0.3779  0.0049    0.7592   0.0039    0.5575       0.5589          0.4120
            TabPFN   0.3766     NaN    0.7880      NaN    0.5377       0.6038          0.4147
   DummyClassifier   0.2233  0.0000    0.5004   0.0000    0.3325       0.3333          0.3335


## 4.1 F1 Macro Comparison

In [14]:
# F1 Macro: Bar Chart with Error Bars
model_order = df_summary.sort_values('f1_mean')['model'].tolist()
 
fig = px.bar(
    df_summary.sort_values('f1_mean'),
    x='f1_mean', y='model', error_x='f1_std',
    orientation='h',
    text=df_summary.sort_values('f1_mean').apply(
        lambda r: f"{r['f1_mean']:.1%}"
        if pd.isna(r['f1_std'])
        else f"{r['f1_mean']:.1%} ± {r['f1_std']:.1%}",  
        axis=1
    ),
    title=f'Benchmark: F1 Macro ({N_LOOPS} Runs, n_samples={N_SAMPLES})<br>'
          f'<sup>RF_tuned & XGB_tuned: single run on full training data (no std)</sup>',
    labels={'f1_mean': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig.update_layout(
    xaxis_tickformat='.0%',
    showlegend=False,
    yaxis=dict(categoryorder='array', categoryarray=model_order),
    height=450,  
)
show_and_save(fig, "benchmark_f1_macro_comparison")

21:29:45 | Chromium init'ed with kwargs {}
21:29:45 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:29:45 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpqwnyxdlk.
21:29:45 | Opening browser.
21:29:45 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp16dmm9ro.
21:29:45 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp16dmm9ro
21:29:45 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpqwnyxdlk/index.html
21:29:45 | Waiting on all navigates
21:29:45 | All navigates done, putting them all in queue.
21:29:47 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpqwnyxdlk/index.html
21:29:47 | Waiting on all navigates
21:29:47 | All navigates done, putting them all in queue.
21:29:47 | Tab ready: 9BCB485938E8A633476267879C5B3B11
21:29:47 | Getting tab from queue (has 1)
21:29:47 | Got 9BCB
21:29:47 | Processing Benchma

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/benchmark_f1_macro_comparison.png


In [15]:
# Filter to models with multiple runs only
multi_run_models = df_results.groupby('model').filter(lambda x: len(x) > 1)
 
fig2 = px.violin(
    multi_run_models, x='model', y='f1_macro', box=True, points='all',
    title=f'F1 Macro Distribution ({N_LOOPS} Runs per Model)<br>'
          f'<sup>RF_tuned & XGB_tuned excluded — single-run models trained on full data</sup>',  # ← CHANGED
    labels={'f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white',
    category_orders={'model': [m for m in model_order[::-1] if m in multi_run_models['model'].unique()]}
)
fig2.update_layout(yaxis_tickformat='.0%', showlegend=False, height=500)
show_and_save(fig2, "benchmark_f1_violin")

21:29:47 | TemporaryDirectory.cleanup() worked.
21:29:47 | shutil.rmtree worked.
21:29:47 | Chromium init'ed with kwargs {}
21:29:47 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:29:47 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmph27c3rl6.
21:29:47 | Opening browser.
21:29:47 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmph3t6a8zb.
21:29:47 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmph3t6a8zb
21:29:48 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmph27c3rl6/index.html
21:29:48 | Waiting on all navigates
21:29:48 | All navigates done, putting them all in queue.
21:29:49 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmph27c3rl6/index.html
21:29:49 | Waiting on all navigates
21:29:49 | All navigates done, putting them all in queue.
21:29:49 | Tab ready: DD00556CAD7D85C13C1C75910B53EFF9
21:29:49 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/benchmark_f1_violin.png


## 4.2 ROC-AUC Comparison

In [16]:
# AUC + F1 Combined: Grouped Bar
metrics_long = []
for _, row in df_summary.iterrows():
    metrics_long.append({'model': row['model'], 'Metric': 'F1 Macro', 'Score': row['f1_mean'], 'Std': row['f1_std']})
    metrics_long.append({'model': row['model'], 'Metric': 'ROC-AUC', 'Score': row['auc_mean'], 'Std': row['auc_std']})
    metrics_long.append({'model': row['model'], 'Metric': 'Accuracy', 'Score': row['acc_mean'], 'Std': row['acc_std']})

ml_df = pd.DataFrame(metrics_long)

fig3 = px.bar(
    ml_df, x='model', y='Score', color='Metric', error_y='Std',
    barmode='group',
    title=f'Benchmark: All Metrics Comparison<br>'
          f'<sup>F1 Macro, ROC-AUC & Accuracy for all models</sup)',
    template='plotly_white',
    text_auto='.1%',
    color_discrete_map={'F1 Macro': '#e74c3c', 'ROC-AUC': '#3498db', 'Accuracy': '#2ecc71'},
    category_orders={'model': model_order[::-1]}
)
fig3.update_layout(yaxis_tickformat='.0%', xaxis_title=None, height=500)
show_and_save(fig3, "benchmark_all_metrics")

21:29:50 | TemporaryDirectory.cleanup() worked.
21:29:50 | shutil.rmtree worked.
21:29:50 | Chromium init'ed with kwargs {}
21:29:50 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:29:50 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_5c2aa52.
21:29:50 | Opening browser.
21:29:50 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmphklhhguw.
21:29:50 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmphklhhguw
21:29:50 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_5c2aa52/index.html
21:29:50 | Waiting on all navigates
21:29:50 | All navigates done, putting them all in queue.
21:29:51 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_5c2aa52/index.html
21:29:51 | Waiting on all navigates
21:29:52 | All navigates done, putting them all in queue.
21:29:52 | Tab ready: C09842AFBBF1B92397A91A4FBD4777FF
21:29:52 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/benchmark_all_metrics.png


## 4.3 Per-Class F1 Comparison

In [17]:
# Per-Class F1: Heatmap
class_cols = {
    'Early (<1Y)': 'f1_early_mean',
    'Late (>1Y - 3Y)': 'f1_late_mean',
    'Healthy': 'f1_healthy_mean'
}

# Build matrix
heatmap_data = []
models_sorted = df_summary.sort_values('f1_mean', ascending=False)['model'].tolist()

for model in models_sorted:
    row = df_summary[df_summary['model'] == model].iloc[0]
    heatmap_data.append([row[col] for col in class_cols.values()])

z = np.array(heatmap_data)
annot = [[f"{v:.1%}" for v in row] for row in z]

fig4 = ff.create_annotated_heatmap(
    z,
    x=list(class_cols.keys()),
    y=models_sorted,
    annotation_text=annot,
    colorscale='RdYlGn',
    showscale=True
)
fig4.update_layout(
    title='Per-Class F1 Score by Model',
    template='plotly_white',
    height=400
)
show_and_save(fig4, "benchmark_per_class_heatmap")

21:29:52 | Chromium init'ed with kwargs {}
21:29:52 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:29:52 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp4yd8qd_y.
21:29:52 | Opening browser.
21:29:52 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpu3zyybnf.
21:29:52 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpu3zyybnf
21:29:52 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp4yd8qd_y/index.html
21:29:52 | Waiting on all navigates
21:29:52 | All navigates done, putting them all in queue.
21:29:54 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp4yd8qd_y/index.html
21:29:54 | Waiting on all navigates
21:29:54 | All navigates done, putting them all in queue.
21:29:54 | Tab ready: 0C22CCF9EA8381B09C3D4E9A1CEE056E
21:29:54 | Getting tab from queue (has 1)
21:29:54 | Got 0C22
21:29:54 | Processing Per_Cla

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/benchmark_per_class_heatmap.png


# 5. Final Test Evaluation

Best model per algorithm (highest F1 on val) evaluated once on the **test set**.

In [18]:
from sklearn.base import clone

test_results = []

# sklearn Modelle
for model_name, pipeline in MODELS.items():
    df_model = df_results[df_results['model'] == model_name]
    best_row = df_model.loc[df_model['f1_macro'].idxmax()]
    best_seed = int(best_row['seed'])

    df_bal_best = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr_best, y_tr_best = get_X_y(df_bal_best)
    clf_test = clone(MODELS[model_name])
    clf_test.fit(X_tr_best, y_tr_best)
    y_test_pred = clf_test.predict(X_test)
    y_test_proba = clf_test.predict_proba(X_test)

    f1_pc = f1_score(y_test, y_test_pred, average=None)
    test_results.append({
        'model': model_name, 'best_seed': best_seed,
        'val_f1': best_row['f1_macro'],
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'test_f1_macro': f1_score(y_test, y_test_pred, average='macro'),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro'),
        'test_f1_early': f1_pc[0], 'test_f1_late': f1_pc[1], 'test_f1_healthy': f1_pc[2],
    })
    log.info(f"\n📊 {model_name} (seed={best_seed}):")
    log.info(f"   Test: Acc={test_results[-1]['test_accuracy']:.2%} | F1={test_results[-1]['test_f1_macro']:.2%} | AUC={test_results[-1]['test_roc_auc']:.2%}")
    log.info(classification_report(y_test, y_test_pred, target_names=['Early (<1Y)', 'Late (>1Y - 3Y)', 'Healthy']))

# TabPFN via subprocess
tabpfn_val_seed = int(df_results[df_results['model'] == 'TabPFN'].iloc[0]['seed'])
tabpfn_test_path = str(RESULTS_DIR / "tabpfn_test_result.json")
log.info(f"Starte TabPFN Test subprocess (seed={tabpfn_val_seed})...")
proc = subprocess.run([
    sys.executable,
    "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabpfn_test.py",
    tabpfn_test_path, str(tabpfn_val_seed), str(N_SAMPLES)
])
if proc.returncode == 0:
    tabpfn_test = json.load(open(tabpfn_test_path))
    tabpfn_test['val_f1'] = float(df_results[df_results['model'] == 'TabPFN'].iloc[0]['f1_macro'])
    test_results.append(tabpfn_test)
    log.info(f"\n📊 TabPFN (seed={tabpfn_val_seed}):")
    log.info(f"   Test: F1={tabpfn_test['test_f1_macro']:.2%} | AUC={tabpfn_test['test_roc_auc']:.2%}")
else:
    log.error("❌ TabPFN Test subprocess error")

# TabICL via subprocess
tabicl_test_path = str(RESULTS_DIR / "tabicl_test_result.json")
log.info(f"Starte TabICL Test subprocess...")
proc_icl = subprocess.run([
    sys.executable,
    "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabicl_test.py",
    tabicl_test_path, str(42), str(N_SAMPLES)
])
if proc_icl.returncode == 0:
    tabicl_test = json.load(open(tabicl_test_path))
    tabicl_test['val_f1'] = float(df_results[df_results['model'] == 'TabICL'].iloc[0]['f1_macro'])
    test_results.append(tabicl_test)
    log.info(f"\n📊 TabICL: F1={tabicl_test['test_f1_macro']:.2%} | AUC={tabicl_test['test_roc_auc']:.2%}")
else:
    log.error("❌ TabICL Test subprocess error")

# RF_tuned & XGB_tuned — already fitted on full data, no retraining needed
for tuned_name, tuned_model, tuned_result in [
    ("RF_tuned",  rf_final,  rf_tuned_result),
    ("XGB_tuned", xgb_final, xgb_tuned_result),
]:
    y_test_pred  = tuned_model.predict(X_test)
    y_test_proba = tuned_model.predict_proba(X_test)
    f1_pc = f1_score(y_test, y_test_pred, average=None)
    test_results.append({
        'model':           tuned_name,
        'best_seed':       42,
        'val_f1':          tuned_result['f1_macro'],
        'test_accuracy':   accuracy_score(y_test, y_test_pred),
        'test_f1_macro':   f1_score(y_test, y_test_pred, average='macro'),
        'test_roc_auc':    roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro'),
        'test_f1_early':   f1_pc[0],
        'test_f1_late':    f1_pc[1],
        'test_f1_healthy': f1_pc[2],
    })
    log.info(f"\n📊 {tuned_name}:")
    log.info(f"   Test: Acc={test_results[-1]['test_accuracy']:.2%} | F1={test_results[-1]['test_f1_macro']:.2%} | AUC={test_results[-1]['test_roc_auc']:.2%}")
    log.info(classification_report(y_test, y_test_pred, target_names=['Early (<1Y)', 'Late (>1Y - 3Y)', 'Healthy']))
 

df_test_results = pd.DataFrame(test_results).sort_values('test_f1_macro', ascending=False)
df_test_results.to_csv(RESULTS_DIR / "test_final_results.csv", index=False)
log.info("\n" + df_test_results.to_string(index=False))

21:29:55 | TemporaryDirectory.cleanup() worked.
21:29:55 | shutil.rmtree worked.
21:29:55 | TemporaryDirectory.cleanup() worked.
21:29:55 | shutil.rmtree worked.
21:29:55 | 
📊 DummyClassifier (seed=42):
21:29:55 |    Test: Acc=33.19% | F1=22.25% | AUC=49.92%
21:29:55 |                  precision    recall  f1-score   support

    Early (<1Y)       0.07      0.33      0.11      3013
Late (>1Y - 3Y)       0.04      0.33      0.07      1856
        Healthy       0.89      0.33      0.48     39822

       accuracy                           0.33     44691
      macro avg       0.33      0.33      0.22     44691
   weighted avg       0.80      0.33      0.44     44691

21:29:55 | 
📊 LogisticRegression (seed=43):
21:29:55 |    Test: Acc=57.59% | F1=39.11% | AUC=76.80%
21:29:55 |                  precision    recall  f1-score   support

    Early (<1Y)       0.21      0.59      0.31      3013
Late (>1Y - 3Y)       0.08      0.53      0.14      1856
        Healthy       0.96      0.58      0.7

Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting test...
proba test...
✅ F1=0.4003 | AUC=0.8099


21:34:41 | 
📊 TabPFN (seed=42):
21:34:41 |    Test: F1=40.03% | AUC=80.99%
21:34:41 | Starte TabICL Test subprocess...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting test...
proba test...
✅ F1=0.4006 | AUC=0.8092


21:39:14 | 
📊 TabICL: F1=40.06% | AUC=80.92%
21:39:15 | 
📊 RF_tuned:
21:39:15 |    Test: Acc=64.62% | F1=43.43% | AUC=81.39%
21:39:15 |                  precision    recall  f1-score   support

    Early (<1Y)       0.21      0.61      0.31      3013
Late (>1Y - 3Y)       0.13      0.63      0.22      1856
        Healthy       0.96      0.65      0.78     39822

       accuracy                           0.65     44691
      macro avg       0.43      0.63      0.43     44691
   weighted avg       0.88      0.65      0.72     44691

21:39:16 | 
📊 XGB_tuned:
21:39:16 |    Test: Acc=61.42% | F1=42.33% | AUC=82.17%
21:39:16 |                  precision    recall  f1-score   support

    Early (<1Y)       0.21      0.64      0.31      3013
Late (>1Y - 3Y)       0.12      0.70      0.21      1856
        Healthy       0.97      0.61      0.75     39822

       accuracy                           0.61     44691
      macro avg       0.43      0.65      0.42     44691
   weighted avg       0.89

In [19]:
# --- Test Results: Bar Chart  ---
df_test_sorted = df_test_results.sort_values('test_f1_macro')

fig5 = px.bar(
    df_test_sorted,
    x='test_f1_macro', y='model',
    orientation='h',
    text=df_test_sorted.apply(
        lambda r: f"F1={r['test_f1_macro']:.1%} | AUC={r['test_roc_auc']:.1%}", axis=1
    ),
    title='Final Test Set: F1 Macro by Model',
    labels={'test_f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig5.update_layout(xaxis_tickformat='.0%', showlegend=False, height=400)
show_and_save(fig5, "benchmark_test_f1")

21:39:16 | Chromium init'ed with kwargs {}
21:39:16 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:39:16 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp914_i4us.
21:39:16 | Opening browser.
21:39:16 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp1q3i0zib.
21:39:16 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp1q3i0zib
21:39:16 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp914_i4us/index.html
21:39:16 | Waiting on all navigates
21:39:16 | All navigates done, putting them all in queue.
21:39:18 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp914_i4us/index.html
21:39:18 | Waiting on all navigates
21:39:18 | All navigates done, putting them all in queue.
21:39:18 | Tab ready: E4375FD05A188FABDE1B117FF041668E
21:39:18 | Getting tab from queue (has 1)
21:39:18 | Got E437
21:39:18 | Processing Final_T

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/benchmark_test_f1.png


# 6. ROC Curves (Test Set, Best Models)

One-vs-Rest ROC curves for each model on the test set, all in one plot per class.

In [20]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.base import clone
from pathlib import Path

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
class_names = ['Early (<1Y)', 'Late (>1Y - 3Y)', 'Healthy']
model_colors = {
    'DummyClassifier': '#95a5a6',
    'LogisticRegression': '#3498db',
    'RandomForest': '#2ecc71',
    'XGBoost': '#e67e22',
    'TabPFN': '#e74c3c',
    'TabICL': '#9b59b6',
    'RF_tuned': '#1a8a4a',  
    'XGB_tuned':'#c0392b', 

}

# sklearn Modelle neu trainieren
best_probas = {}
for model_name, pipeline in MODELS.items():
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])
    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    clf = clone(MODELS[model_name])
    clf.fit(X_tr, y_tr)
    best_probas[model_name] = clf.predict_proba(X_test)

# TabPFN proba aus Subprocess-Datei laden
tabpfn_proba_path = tabpfn_test_path.replace('.json', '_proba.npy')
if Path(tabpfn_proba_path).exists():
    best_probas['TabPFN'] = np.load(tabpfn_proba_path)
    log.info("TabPFN proba loaded")
else:
    log.warning("⚠️ TabPFN not found")

tabicl_proba_path = tabicl_test_path.replace('.json', '_proba.npy')
if Path(tabicl_proba_path).exists():
    best_probas['TabICL'] = np.load(tabicl_proba_path)
    log.info("TabICL proba loaded")
else:
    log.warning("⚠️ TabICL not found")

# tuned models already fitted, just predict_proba
best_probas['RF_tuned']  = rf_final.predict_proba(X_test)
best_probas['XGB_tuned'] = xgb_final.predict_proba(X_test)

# Plot
from plotly.subplots import make_subplots

fig6 = make_subplots(rows=1, cols=3, subplot_titles=[f'OvR: {cn}' for cn in class_names])

for i, cn in enumerate(class_names):
    for model_name, proba in best_probas.items():
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
        roc_auc_val = auc(fpr, tpr)
        fig6.add_trace(
            go.Scatter(x=fpr, y=tpr, mode='lines',
                       name=f'{model_name} ({roc_auc_val:.3f})',
                       line=dict(color=model_colors.get(model_name, 'grey'), width=2),
                       showlegend=(i == 0)),
            row=1, col=i+1
        )
    fig6.add_trace(
        go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                   line=dict(color='grey', width=1, dash='dash'), showlegend=False),
        row=1, col=i+1
    )

fig6.update_layout(
    title='ROC Curves (Test Set) — One-vs-Rest per Class',
    template='plotly_white', height=600, width=1600,
    legend=dict(x=1.02, y=1)
)
for i in range(3):
    fig6.update_xaxes(title_text='FPR', row=1, col=i+1)
    fig6.update_yaxes(title_text='TPR', row=1, col=i+1)

show_and_save(fig6, "benchmark_roc_curves")

21:39:19 | TemporaryDirectory.cleanup() worked.
21:39:19 | shutil.rmtree worked.
21:39:20 | TabPFN proba loaded
21:39:20 | TabICL proba loaded
21:39:20 | Chromium init'ed with kwargs {}
21:39:20 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:39:20 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_e_0pp_0.
21:39:20 | Opening browser.
21:39:20 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp8ahhs47l.
21:39:20 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp8ahhs47l
21:39:20 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_e_0pp_0/index.html
21:39:20 | Waiting on all navigates
21:39:20 | All navigates done, putting them all in queue.
21:39:22 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_e_0pp_0/index.html
21:39:22 | Waiting on all navigates
21:39:22 | All navigates done, putting them all in queue.
21:

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/benchmark_roc_curves.png


# 7. Data Quality Check: Dead but "Healthy"?

Patients classified as **target=2 (healthy/censored)** who have a recorded date of death (`dod`). These patients died but were never diagnosed with heart failure — they are **correctly** censored (not false labels), but it's important to know how many there are and whether they bias the model.

In [21]:
# ── Data Quality: Deceased Patients in the "Healthy" Class ──
import polars as pl

# df is the full dataset (before the split)
df_quality = df.to_pandas() if hasattr(df, 'to_pandas') else df

# Patients with target=2 (healthy/censored) AND date of death (dod) present
dead_but_healthy = df.filter(
    (pl.col("target") == 2) & (pl.col("dod").is_not_null())
)

total_healthy = df.filter(pl.col("target") == 2).height
n_dead_healthy = dead_but_healthy.height

log.info(f"Total patients with target=2 (healthy/censored): {total_healthy:,}")
log.info(f"With date of death (dod): {n_dead_healthy:,} ({n_dead_healthy/total_healthy:.1%})")
log.info(f"Without dod (truly alive): {total_healthy - n_dead_healthy:,}")

# Distribution of survival time for the 'dead healthy' patients
if n_dead_healthy > 0:
    t_death_stats = dead_but_healthy.select("t_death").to_pandas()["t_death"].describe()
    log.info(f"\nSurvival time (t_death) of deceased 'healthy' patients:")
    log.info(t_death_stats)
    
    fig_dq = px.histogram(
        dead_but_healthy.select("t_death").to_pandas(), 
        x="t_death", nbins=50,
        title=f"Deceased patients without HF diagnosis (n={n_dead_healthy:,}): Days until death",
        labels={"t_death": "Days from baseline to death"},
        template="plotly_white"
    )
    fig_dq.add_vline(x=365, line_dash="dash", line_color="red", annotation_text="1 year")
    show_and_save(fig_dq, "data_quality_dead_but_healthy")

21:39:23 | Total patients with target=2 (healthy/censored): 199,110
21:39:23 | With date of death (dod): 30,236 (15.2%)
21:39:23 | Without dod (truly alive): 168,874
21:39:23 | 
Survival time (t_death) of deceased 'healthy' patients:
21:39:23 | count    30236.00000
mean       661.31952
std        978.55315
min      -2768.00000
25%         39.00000
50%        218.00000
75%        870.25000
max       5615.00000
Name: t_death, dtype: float64
21:39:23 | Chromium init'ed with kwargs {}
21:39:23 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:39:23 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpn409s4a4.
21:39:23 | Opening browser.
21:39:23 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpk_p5tqng.
21:39:23 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpk_p5tqng
21:39:24 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpn409s4a4/index.ht

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/data_quality_dead_but_healthy.png


# 8. Correlation vs. Causality (Feature Analysis)

**Philipp's feedback: "Remove confounders"** — Identify features with correlation but no causality.

| Analysis | What it measures | Method |
|----------|------------------|--------|
| **Correlation** (univariate) | How strongly does a feature *alone* relate to the target? | Cramér's V (categorical), Eta² (numerical) |
| **Predictive importance** (multivariate) | How much *unique* predictive power does a feature have, when all others are known? | Permutation Importance (F1 Macro) |

**Interpretation:**
- High correlation + high importance → **True driver** (e.g., age, BMI)
- High correlation + low/negative importance → **Confounder** (e.g., insurance correlates with age)
- Low correlation + low importance → **Irrelevant** (can be removed)


## 8.1 Univariate Correlation (Feature ↔ Target)

In [22]:
# Univariate Correlation: How strongly does each feature alone relate to the target?
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Cramér's V: Association for categorical × categorical (0 = none, 1 = perfect)."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = len(x)
    min_dim = min(ct.shape) - 1
    if min_dim == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))

def eta_squared(feature_values, target_values):
    """Eta²: Effect size for numerical × categorical (ANOVA). 0 = none, 1 = perfect."""
    groups = [feature_values[target_values == c].dropna() for c in sorted(target_values.unique())]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
   
    grand_mean = feature_values.dropna().mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = ((feature_values.dropna() - grand_mean)**2).sum()
    if ss_total == 0:
        return 0.0
    return ss_between / ss_total

# Compute correlation for each feature
df_val_pd = X_val.copy()
df_val_pd['target'] = y_val

correlation_results = []
for feature in FEATURE_COLS:
    if feature in CAT_COLS:
        corr = cramers_v(df_val_pd[feature], df_val_pd['target'])
        method = "Cramér's V"
    else:
        corr = eta_squared(df_val_pd[feature], df_val_pd['target'])
        method = "Eta²"
    correlation_results.append({
        'Feature': feature,
        'Correlation': corr,
        'Method': method,
        'Type': 'categorical' if feature in CAT_COLS else 'numerical'
    })

df_corr = pd.DataFrame(correlation_results).sort_values('Correlation', ascending=False)

log.info("\nUnivariate Correlation (Feature → Target):\n")
log.info(df_corr.to_string(index=False, float_format='{:.4f}'.format))

# Plot
fig_corr = px.bar(
    df_corr.sort_values('Correlation'),
    x='Correlation', y='Feature', orientation='h',
    color='Type',
    color_discrete_map={'categorical': '#3498db', 'numerical': '#e74c3c'},
    text=df_corr.sort_values('Correlation').apply(
        lambda r: f"{r['Correlation']:.3f} ({r['Method']})", axis=1
    ),
    title=f"Univariate Correlation: Feature ↔ Target (n={len(y_val)})",
    labels={'Correlation': 'Association Strength', 'Feature': ''},
    template='plotly_white'
)
fig_corr.update_layout(height=450)
show_and_save(fig_corr, "correlation_univariate")

21:39:26 | 
Univariate Correlation (Feature → Target):

21:39:26 |        Feature  Correlation     Method        Type
admission_type       0.1445 Cramér's V categorical
        gender       0.1241 Cramér's V categorical
     insurance       0.1233 Cramér's V categorical
          race       0.0913 Cramér's V categorical
marital_status       0.0816 Cramér's V categorical
    anchor_age       0.0571       Eta²   numerical
      language       0.0532 Cramér's V categorical
           bmi       0.0016       Eta²   numerical
21:39:26 | TemporaryDirectory.cleanup() worked.
21:39:26 | shutil.rmtree worked.
21:39:26 | Chromium init'ed with kwargs {}
21:39:26 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:39:26 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpwswumw4s.
21:39:26 | Opening browser.
21:39:26 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpn0y0qavu.
21:39:26 | Temporary directory at: 

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/correlation_univariate.png


## 8.2 Predictive Importance (Permutation Importance)

Measures how much **unique** predictive power a feature has **when all other features are known**. A feature with high correlation but low importance is a confounder — it only correlates because it is associated with a true driver.

In [23]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone

importance_results = []

for model_name in MODELS.keys():
    log.info(f"\n🔄 {model_name}...")
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])

    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    clf = clone(MODELS[model_name])
    clf.fit(X_tr, y_tr)
    perm = permutation_importance(clf, X_val, y_val, n_repeats=10,
                                   random_state=42, scoring='f1_macro', n_jobs=-1)
    for j, feature in enumerate(FEATURE_COLS):
        importance_results.append({
            'Model': model_name, 'Feature': feature,
            'Importance': perm.importances_mean[j],
            'Std': perm.importances_std[j],
        })
    log.info(f"  ✅ done")


log.info("TabPFN Permutation Importance skipped (35k Val × 10 repeats)")
for feature in FEATURE_COLS:
    importance_results.append({
        'Model': 'TabPFN', 'Feature': feature,
        'Importance': float('nan'), 'Std': float('nan')
    })

log.info("TabICL Permutation Importance skipped")
for feature in FEATURE_COLS:
    importance_results.append({
        'Model': 'TabICL', 'Feature': feature,
        'Importance': float('nan'), 'Std': float('nan')
    })
# tuned models — use rf_final/xgb_final directly, no retraining
for tuned_name, tuned_model in [("RF_tuned", rf_final), ("XGB_tuned", xgb_final)]:
    log.info(f"\n🔄 {tuned_name}...")
    perm = permutation_importance(
        tuned_model, X_val, y_val,
        n_repeats=10, random_state=42, scoring='f1_macro', n_jobs=-1
    )
    for j, feature in enumerate(FEATURE_COLS):
        importance_results.append({
            'Model':      tuned_name,
            'Feature':    feature,
            'Importance': perm.importances_mean[j],
            'Std':        perm.importances_std[j],
        })
    log.info(f"  ✅ done")

df_importance = pd.DataFrame(importance_results)
df_importance.to_csv(RESULTS_DIR / "permutation_importance_all_models.csv", index=False)

df_imp_avg = df_importance[
    ~df_importance['Model'].isin(['DummyClassifier', 'TabPFN', 'TabICL'])
].groupby('Feature').agg(
    Importance_mean=('Importance', 'mean'),
    Importance_std=('Importance', 'std'),
).reset_index().sort_values('Importance_mean', ascending=False)

log.info("\n📊 Average Permutation Importance (excluding Dummy + TabPFN):\n")
log.info(df_imp_avg.to_string(index=False, float_format='{:.4f}'.format))

21:39:28 | 
🔄 DummyClassifier...
21:39:31 |   ✅ done
21:39:31 | 
🔄 LogisticRegression...
21:39:32 |   ✅ done
21:39:32 | 
🔄 RandomForest...
21:39:32 | TemporaryDirectory.cleanup() worked.
21:39:32 | shutil.rmtree worked.
21:39:40 |   ✅ done
21:39:40 | 
🔄 XGBoost...
21:39:44 |   ✅ done
21:39:44 | TabPFN Permutation Importance skipped (35k Val × 10 repeats)
21:39:44 | TabICL Permutation Importance skipped
21:39:44 | 
🔄 RF_tuned...
21:40:12 |   ✅ done
21:40:12 | 
🔄 XGB_tuned...
21:40:23 |   ✅ done
21:40:24 | 
📊 Average Permutation Importance (excluding Dummy + TabPFN):

21:40:24 |        Feature  Importance_mean  Importance_std
    anchor_age           0.0434          0.0146
admission_type           0.0414          0.0043
           bmi           0.0241          0.0141
        gender           0.0088          0.0017
          race           0.0028          0.0018
      language           0.0014          0.0006
marital_status           0.0008          0.0036
     insurance           0.0007 

In [24]:
# ── Importance Heatmap: pro Modell × Feature ──
pivot = df_importance.pivot_table(index='Model', columns='Feature', values='Importance')
models_order = ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost','RF_tuned', 'XGB_tuned', 'TabPFN', 'TabICL']
pivot = pivot.reindex(models_order)
features_order = df_imp_avg.sort_values('Importance_mean', ascending=True)['Feature'].tolist()
pivot = pivot[features_order]

annot = [[f"{v:.3f}" for v in row] for row in pivot.values]

fig_imp_heat = ff.create_annotated_heatmap(
    pivot.values, x=features_order, y=models_order,
    annotation_text=annot, colorscale='RdBu', reversescale=False, showscale=True
)
fig_imp_heat.update_layout(
    title='Permutation Importance: Feature × Model (F1 Macro)',
    template='plotly_white', height=400
)
show_and_save(fig_imp_heat, "importance_heatmap_all_models")

21:40:24 | Chromium init'ed with kwargs {}
21:40:24 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:40:24 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_t1122q7.
21:40:24 | Opening browser.
21:40:24 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpw2nwa53w.
21:40:24 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpw2nwa53w
21:40:24 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_t1122q7/index.html
21:40:24 | Waiting on all navigates
21:40:24 | All navigates done, putting them all in queue.
21:40:25 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_t1122q7/index.html
21:40:25 | Waiting on all navigates
21:40:26 | All navigates done, putting them all in queue.
21:40:26 | Tab ready: 3C3E479E48634E03EC6CF26BFA3A2940
21:40:26 | Getting tab from queue (has 1)
21:40:26 | Got 3C3E
21:40:26 | Processing Permuta

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/importance_heatmap_all_models.png


## 8.3 Correlation vs. Causality — The Confounder Plot

Compares univariate correlation (descriptive) with multivariate importance (predictive). Features in the **upper left quadrant** (high correlation, low importance) are "confounders" — they correlate because they are proxies for true drivers.

In [25]:
# --- Correlation vs. Causality: Scatter Plot ---
df_combined = df_corr.merge(df_imp_avg, on='Feature')

# Feature classification
def classify_feature(row):
    corr_threshold = df_combined['Correlation'].median()
    imp_threshold = 0.005  # minimal positive importance
    if row['Correlation'] >= corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '✅ True driver'
    elif row['Correlation'] >= corr_threshold and row['Importance_mean'] < imp_threshold:
        return '⚠️ Confounder'
    elif row['Correlation'] < corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '🔍 Hidden driver'
    else:
        return '❌ Irrelevant'

df_combined['Category'] = df_combined.apply(classify_feature, axis=1)

# Scatter: Correlation (x) vs. Importance (y)
fig_scatter = px.scatter(
    df_combined, 
    x='Correlation', y='Importance_mean',
    text='Feature',
    color='Category',
    color_discrete_map={
        '✅ True driver': '#2ecc71',
        '⚠️ Confounder': '#e67e22',
        '🔍 Hidden driver': '#3498db',
        '❌ Irrelevant': '#95a5a6',
    },
    error_y='Importance_std',
    title='Correlation vs. Predictive Importance — "Confounder Plot"',
    labels={
        'Correlation': 'Univariate Correlation (Cramér\'s V / Eta²)',
        'Importance_mean': 'Permutation Importance (mean across models, F1 Macro)'
    },
    template='plotly_white'
)

# Quadrant lines
corr_median = df_combined['Correlation'].median()
fig_scatter.add_hline(y=0.005, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Importance threshold")
fig_scatter.add_vline(x=corr_median, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Correlation median")
fig_scatter.add_hline(y=0, line_dash="solid", line_color="black", opacity=0.3)

fig_scatter.update_traces(textposition='top center', marker=dict(size=14))
fig_scatter.update_layout(height=600, width=900)
show_and_save(fig_scatter, "correlation_vs_causality_scatter", width=900)


21:40:26 | Chromium init'ed with kwargs {}
21:40:26 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:40:26 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp5bx95fv7.
21:40:26 | Opening browser.
21:40:26 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp2x2o0ae4.
21:40:26 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp2x2o0ae4
21:40:26 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp5bx95fv7/index.html
21:40:26 | Waiting on all navigates
21:40:26 | All navigates done, putting them all in queue.
21:40:28 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp5bx95fv7/index.html
21:40:28 | Waiting on all navigates
21:40:28 | All navigates done, putting them all in queue.
21:40:28 | Tab ready: 1EE3150573E683EA82B6E0CAF032527D
21:40:28 | Getting tab from queue (has 1)
21:40:28 | Got 1EE3
21:40:28 | Processing Correla

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/correlation_vs_causality_scatter.png


In [26]:
df_side = df_combined.sort_values('Correlation', ascending=True)

fig_side = go.Figure()
fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Correlation'],
    name='Correlation (univariate)', orientation='h',
    marker_color='#3498db', opacity=0.8
))
fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Importance_mean'],
    name='Importance (multivariate)', orientation='h',
    marker_color='#e74c3c', opacity=0.8,
    error_x=dict(type='data', array=df_side['Importance_std'].values, visible=True)
))
fig_side.update_layout(
    barmode='group',
    title='Correlation vs. Predictive Importance (Side-by-Side)',
    xaxis_title='Strength',
    yaxis_title=None,
    template='plotly_white',
    height=500,
    legend=dict(x=0.6, y=0.05)
)
show_and_save(fig_side, "correlation_vs_importance_bars")

log.info("\n📊 Feature Classification:\n")
display_df = df_combined[['Feature', 'Correlation', 'Method', 'Importance_mean', 'Importance_std', 'Category']]
display_df = display_df.sort_values('Correlation', ascending=False)
log.info(display_df.to_string(index=False, float_format='{:.4f}'.format))

df_combined.to_csv(RESULTS_DIR / "correlation_vs_importance.csv", index=False)

log.info("\n" + "="*60)
true_drivers = df_combined[df_combined['Category'].str.contains('True driver')]
confounders  = df_combined[df_combined['Category'].str.contains('Confounder')]
irrelevant   = df_combined[df_combined['Category'].str.contains('Irrelevant')]

if not true_drivers.empty:
    log.info(f"✅ True drivers: {', '.join(true_drivers['Feature'].tolist())}")
if not confounders.empty:
    log.info(f"⚠️  Confounders: {', '.join(confounders['Feature'].tolist())}")
    log.info(f"   → Correlate with target but no unique predictive power.")
if not irrelevant.empty:
    log.info(f"❌ Irrelevant: {', '.join(irrelevant['Feature'].tolist())}")
log.info("="*60)

21:40:29 | Chromium init'ed with kwargs {}
21:40:29 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:40:29 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpxtay6h1v.
21:40:29 | Opening browser.
21:40:29 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmppngwnl_c.
21:40:29 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmppngwnl_c
21:40:29 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpxtay6h1v/index.html
21:40:29 | Waiting on all navigates
21:40:29 | All navigates done, putting them all in queue.
21:40:30 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpxtay6h1v/index.html
21:40:30 | Waiting on all navigates
21:40:31 | All navigates done, putting them all in queue.
21:40:31 | Tab ready: 281DD02D4C42E4C287556F8C9863F213
21:40:31 | Getting tab from queue (has 1)
21:40:31 | Got 281D
21:40:31 | Processing Correla

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/correlation_vs_importance_bars.png


21:40:31 | 
📊 Feature Classification:

21:40:31 |        Feature  Correlation     Method  Importance_mean  Importance_std        Category
admission_type       0.1445 Cramér's V           0.0414          0.0043   ✅ True driver
        gender       0.1241 Cramér's V           0.0088          0.0017   ✅ True driver
     insurance       0.1233 Cramér's V           0.0007          0.0049   ⚠️ Confounder
          race       0.0913 Cramér's V           0.0028          0.0018   ⚠️ Confounder
marital_status       0.0816 Cramér's V           0.0008          0.0036    ❌ Irrelevant
    anchor_age       0.0571       Eta²           0.0434          0.0146 🔍 Hidden driver
      language       0.0532 Cramér's V           0.0014          0.0006    ❌ Irrelevant
           bmi       0.0016       Eta²           0.0241          0.0141 🔍 Hidden driver
21:40:31 | 
21:40:31 | ✅ True drivers: admission_type, gender
21:40:31 | ⚠️  Confounders: insurance, race
21:40:31 |    → Correlate with target but no unique 

# 9.1 Category Breakdown (Target Distribution)
Deep Dive into the target classes to identify Hidden & True Drivers.

In [28]:
import plotly.express as px
import pandas as pd

log.info("Generating deep-dive category breakdowns...")

df_eval = X_val.copy()
df_eval['Target'] = y_val
df_eval['Target_Label'] = df_eval['Target'].map({
    0: 'Early Onset (<1Y)', 
    1: 'Late Onset (1-3Y)', 
    2: 'No Event / Healthy'
})

# Take the features requested by the user (gender, bmi) + the most prominent ones from df_combined
features_to_plot = ['gender', 'bmi'] # Requested
for cat in ['True driver', 'Confounder', 'Hidden driver']:
    sub = df_combined[df_combined['Category'].str.contains(cat)]
    if not sub.empty:
        if 'Confounder' in cat:
            top_var = sub.sort_values('Correlation', ascending=False).iloc[0]['Feature']
        else:
            top_var = sub.sort_values('Importance_mean', ascending=False).iloc[0]['Feature']
        if top_var not in features_to_plot:
            features_to_plot.append(top_var)

for feat in features_to_plot:
    # 1. Classify feature (What kind of driver?)
    cat_type = df_combined[df_combined['Feature'] == feat]['Category'].values
    cat_type_str = cat_type[0] if len(cat_type) > 0 else "Manually selected"
    
    clean_cat_str = cat_type_str.replace('✅ ', '').replace('⚠️ ', '').replace('🔍 ', '').replace('❌ ', '')
    
    # 2. Binning if numeric (e.g. bmi, anchor_age)
    if pd.api.types.is_numeric_dtype(df_eval[feat]) and df_eval[feat].nunique() > 10:
        plot_col = f'{feat}_binned'
        # With q=3 (Tertiles) or q=4 (Quartiles). We use q=4 and drop duplicates.
        bins = pd.qcut(df_eval[feat], q=4, duplicates='drop')
        df_eval[plot_col] = bins.astype(str)
    else:
        plot_col = feat
        # Fill Missing or NaN for categories
        df_eval[plot_col] = df_eval[plot_col].fillna('UNKNOWN').astype(str)
        
    # 3. Calculate target class distribution within each category
    dist = df_eval.groupby(plot_col)['Target_Label'].value_counts(normalize=True).reset_index(name='Percent')
    dist['Percent'] = dist['Percent'] * 100
    
    # 4. Identify high/low risk
    early_mask = dist['Target_Label'] == 'Early Onset (<1Y)'
    healthy_mask = dist['Target_Label'] == 'No Event / Healthy'
    
    max_early_row = dist[early_mask].sort_values('Percent', ascending=False).iloc[0] if early_mask.any() else None
    max_healthy_row = dist[healthy_mask].sort_values('Percent', ascending=False).iloc[0] if healthy_mask.any() else None
    
    # Filter out Healthy for the minimalistic plot
    plot_dist = dist[dist['Target_Label'] != 'No Event / Healthy']
    
    # 5. Create Plotly Bar Chart
    fig = px.bar(
        plot_dist, x=plot_col, y='Percent', color='Target_Label',
        barmode='group',
        title=f"<b>{feat.capitalize()}</b><br><sup>{clean_cat_str}</sup>",
        color_discrete_map={
            'Early Onset (<1Y)': '#e74c3c', 
            'Late Onset (1-3Y)': '#f39c12'
        },
        template='plotly_white'
    )
    
    fig.update_layout(
        yaxis_title="Disease Onset Share (%)", 
        xaxis_title=None,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, title=None),
        margin=dict(t=80) 
    )
    
    # Annotate Highest Risk
    if max_early_row is not None:
        fig.add_annotation(
            x=max_early_row[plot_col],
            y=max_early_row['Percent'] + 1.5,
            text=f"⚠️ Elev. Risk",
            showarrow=True,
            arrowhead=1,
            arrowsize=1.5,
            arrowwidth=2,
            arrowcolor="#c0392b",
            font=dict(color="#c0392b", size=11)
        )
        
    # Annotate Lowest Risk (using the category with the most healthy)
    if max_healthy_row is not None:
        safe_cat = max_healthy_row[plot_col]
        # Find the highest plotted bar for this safe category to place the annotation comfortably
        safe_bars = plot_dist[plot_dist[plot_col] == safe_cat]
        safe_y = safe_bars['Percent'].max() if not safe_bars.empty else 0
        
        fig.add_annotation(
            x=safe_cat,
            y=safe_y + 1.5,
            text=f"💡 Mild risk<br>({max_healthy_row['Percent']:.0f}% healthy)",
            showarrow=True,
            arrowhead=1,
            arrowsize=1.5,
            arrowwidth=2,
            arrowcolor="#27ae60",
            font=dict(color="#27ae60", size=11)
        )
        
    show_and_save(fig, f"category_breakdown_{feat}")

21:52:30 | Generating deep-dive category breakdowns...
21:52:30 | TemporaryDirectory.cleanup() worked.
21:52:30 | shutil.rmtree worked.
21:52:30 | Chromium init'ed with kwargs {}
21:52:30 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:52:30 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpj9lakixv.
21:52:30 | Opening browser.
21:52:30 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpgxn54n2n.
21:52:30 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpgxn54n2n
21:52:30 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpj9lakixv/index.html
21:52:30 | Waiting on all navigates
21:52:30 | All navigates done, putting them all in queue.
21:52:32 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpj9lakixv/index.html
21:52:32 | Waiting on all navigates
21:52:32 | All navigates done, putting them all in queue.
21:52:32 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/category_breakdown_gender.png


21:52:32 | Chromium init'ed with kwargs {}
21:52:32 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:52:32 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp3jgxqnm3.
21:52:32 | Opening browser.
21:52:32 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpaa5pb7qe.
21:52:32 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpaa5pb7qe
21:52:33 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp3jgxqnm3/index.html
21:52:33 | Waiting on all navigates
21:52:33 | All navigates done, putting them all in queue.
21:52:34 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp3jgxqnm3/index.html
21:52:34 | Waiting on all navigates
21:52:35 | All navigates done, putting them all in queue.
21:52:35 | Tab ready: C36FC17C45E065CE69584247F866FA79
21:52:35 | Getting tab from queue (has 1)
21:52:35 | Got C36F
21:52:35 | Processing bBmibbr

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/category_breakdown_bmi.png


21:52:35 | Chromium init'ed with kwargs {}
21:52:35 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:52:35 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp74cz5__x.
21:52:35 | Opening browser.
21:52:35 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_6sfhnka.
21:52:35 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_6sfhnka
21:52:35 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp74cz5__x/index.html
21:52:35 | Waiting on all navigates
21:52:35 | All navigates done, putting them all in queue.
21:52:36 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp74cz5__x/index.html
21:52:36 | Waiting on all navigates
21:52:37 | All navigates done, putting them all in queue.
21:52:37 | Tab ready: 1BBEB17263A301F490103EA30E63A684
21:52:37 | Getting tab from queue (has 1)
21:52:37 | Got 1BBE
21:52:37 | Processing bAdmiss

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/category_breakdown_admission_type.png


21:52:37 | TemporaryDirectory.cleanup() worked.
21:52:37 | shutil.rmtree worked.
21:52:37 | TemporaryDirectory.cleanup() worked.
21:52:37 | shutil.rmtree worked.
21:52:37 | Chromium init'ed with kwargs {}
21:52:37 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:52:37 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpegbhru98.
21:52:37 | Opening browser.
21:52:37 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp1fobw73n.
21:52:37 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp1fobw73n
21:52:38 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpegbhru98/index.html
21:52:38 | Waiting on all navigates
21:52:38 | All navigates done, putting them all in queue.
21:52:39 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpegbhru98/index.html
21:52:39 | Waiting on all navigates
21:52:39 | All navigates done, putting the

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/category_breakdown_insurance.png


21:52:40 | TemporaryDirectory.cleanup() worked.
21:52:40 | shutil.rmtree worked.
21:52:40 | Chromium init'ed with kwargs {}
21:52:40 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:52:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_llqdiu3.
21:52:40 | Opening browser.
21:52:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp6rfwr31j.
21:52:40 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp6rfwr31j
21:52:40 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_llqdiu3/index.html
21:52:40 | Waiting on all navigates
21:52:40 | All navigates done, putting them all in queue.
21:52:41 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_llqdiu3/index.html
21:52:41 | Waiting on all navigates
21:52:42 | All navigates done, putting them all in queue.
21:52:42 | Tab ready: 7F4E99BB97857007C31CAE439CC93A9A
21:52:42 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_tuned_20260426_212711/plots/category_breakdown_anchor_age.png


# Summary

This notebook provides four key results for the thesis:

1. **Benchmark** (Section 3-6): TabPFN compared against DummyClassifier, LogisticRegression, RandomForest, and XGBoost — all with identical data pipeline, 20 runs each.

2. **Data Quality** (Section 7): Quantifies how many target=2 patients are actually deceased (censored correctly, but important for interpretation).

3. **Korrelation vs. Kausalität** (Section 8.1–8.3): Systematic analysis distinguishing:
   - **Univariate Korrelation** (Cramér's V / Eta²): deskriptive Assoziation
   - **Permutation Importance** (F1 Macro, alle Modelle): prädiktive Bedeutung
   - **Hosenträger-Plot**: Identifiziert Confounder (hohe Korrelation, keine eigene Vorhersagekraft)


All results are saved in the run folder for reproducibility.